# 01 — Data Ingestion

## About

**Purpose:** Build the labelled modelling dataset by joining NPPES providers with the OIG LEIE exclusion list on NPI.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-14<br>
**Notes:** NPPES is loaded as a 500,000-row sample to keep memory manageable during exploration; LEIE is loaded in full. Join key is NPI.<br>
**Description:** Loads the OIG LEIE exclusion list (~82K excluded providers) and a sample of the NPPES provider registry (330 columns). Checks NPI overlap between the two sources, creates a binary `excluded` target variable (1 if the provider's NPI appears in LEIE, else 0), and saves the labelled dataset to parquet for downstream EDA and modelling.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-05-14 | 1.0     | Ganapathy K | Initial version |

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [2]:
import logging
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd

### 1.2 Configure logging

In [3]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [4]:
nppes_file_path = "D:/Data Science/Datasets/Medical/NPPES/raw/NPPES_Data_Dissemination_March_2026_V2.zip"
nppes_file_name = "npidata_pfile_20050523-20260308.csv"
leie_file_path = Path("D:/Data Science/Visual Studio Code/healthcare-provider-termination/data/raw/oig_leie_202602.csv")
processed_folder_path = Path("D:/Data Science/Visual Studio Code/healthcare-provider-termination/data/processed")

## 2. Data Ingestion
### 2.1 Load OIG LEIE

In [5]:
# Load LEIE — full file, small enough to load entirely
leie_dataframe = pd.read_csv(leie_file_path, encoding='latin-1', low_memory=False)
logger.info(f"LEIE loaded — shape: {leie_dataframe.shape}")
leie_dataframe.head(3)

2026-05-17 02:31:55,874 | INFO | LEIE loaded — shape: (82749, 18)


,LASTNAME,FIRSTNAME,MIDNAME,BUSNAME,GENERAL,SPECIALTY,UPIN,NPI,DOB,ADDRESS,CITY,STATE,ZIP,EXCLTYPE,EXCLDATE,REINDATE,WAIVERDATE,WVRSTATE
0,NaN,NaN,NaN,"#1 MARKETING SERVICE, INC",OTHER BUSINESS,SOBER HOME,NaN,0,NaN,239 BRIGHTON BEACH AVENUE,BROOKLYN,NY,11235,1128a1,20200319,0,0,NaN
1,NaN,NaN,NaN,"1 BEST CARE, INC",OTHER BUSINESS,HOME HEALTH AGENCY,NaN,0,NaN,"2161 UNIVERSITY AVENUE W, STE",SAINT PAUL,MN,55114,1128b5,20230518,0,0,NaN
2,NaN,NaN,NaN,101 FIRST CARE PHARMACY INC,OTHER BUSINESS,PHARMACY,NaN,1972902351,NaN,"C/O 609 W 191ST STREET, APT D",NEW YORK,NY,10040,1128b8,20220320,0,0,NaN


### 2.2 Load NPPES Sample

In [6]:
# Load NPPES — sample of 500K rows to keep memory manageable during exploration
with zipfile.ZipFile(nppes_file_path, 'r') as zip_file:
    with zip_file.open(nppes_file_name) as csv_file:
        nppes_dataframe = pd.read_csv(csv_file, nrows=500_000, low_memory=False)

logger.info(f"NPPES sample loaded — shape: {nppes_dataframe.shape}")
nppes_dataframe.head(3)

2026-05-17 02:32:05,140 | INFO | NPPES sample loaded — shape: (500000, 330)


,NPI,Entity Type Code,Replacement NPI,Employer Identification Number (EIN),Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider Middle Name,Provider Name Prefix Text,Provider Name Suffix Text,...,Healthcare Provider Taxonomy Group_7,Healthcare Provider Taxonomy Group_8,Healthcare Provider Taxonomy Group_9,Healthcare Provider Taxonomy Group_10,Healthcare Provider Taxonomy Group_11,Healthcare Provider Taxonomy Group_12,Healthcare Provider Taxonomy Group_13,Healthcare Provider Taxonomy Group_14,Healthcare Provider Taxonomy Group_15,Certification Date
0,1679576722,1.0,NaN,NaN,NaN,WIEBE,DAVID,A,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1588667638,1.0,NaN,NaN,NaN,PILCHER,WILLIAM,C,DR.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1497758544,2.0,NaN,<UNAVAIL>,"CUMBERLAND COUNTY HOSPITAL SYSTEM, INC",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Target Variable Creation
### 3.1 NPI Overlap Check

In [7]:
# Check NPI overlap between NPPES and LEIE
leie_npis = set(leie_dataframe[leie_dataframe['NPI'] != 0]['NPI'])
nppes_npis = set(nppes_dataframe['NPI'])

matched_npis = leie_npis & nppes_npis

logger.info(f"LEIE records with valid NPI : {len(leie_npis):,}")
logger.info(f"NPPES NPIs in sample        : {len(nppes_npis):,}")
logger.info(f"Matched NPIs                : {len(matched_npis):,}")
logger.info(f"Match rate (LEIE → NPPES)   : {len(matched_npis) / len(leie_npis) * 100:.1f}%")

2026-05-17 02:32:05,219 | INFO | LEIE records with valid NPI : 8,306


2026-05-17 02:32:05,219 | INFO | NPPES NPIs in sample        : 500,000


2026-05-17 02:32:05,220 | INFO | Matched NPIs                : 1,183


2026-05-17 02:32:05,220 | INFO | Match rate (LEIE → NPPES)   : 14.2%


### 3.2 Create Binary Target

In [8]:
# Create binary target variable — excluded = 1 if NPI appears in LEIE, 0 otherwise
nppes_dataframe['excluded'] = nppes_dataframe['NPI'].isin(leie_npis).astype(int)

logger.info(f"Excluded (1) : {nppes_dataframe['excluded'].sum():,}")
logger.info(f"Not excluded (0) : {(nppes_dataframe['excluded'] == 0).sum():,}")
logger.info(f"Class ratio : {nppes_dataframe['excluded'].mean() * 100:.3f}% excluded")

2026-05-17 02:32:05,257 | INFO | Excluded (1) : 1,183


2026-05-17 02:32:05,258 | INFO | Not excluded (0) : 498,817


2026-05-17 02:32:05,259 | INFO | Class ratio : 0.237% excluded


## 4. Outputs / Save
### 4.1 Save Labelled Dataset

In [9]:
# Save the labelled dataset for downstream EDA and modelling
processed_folder_path.mkdir(parents=True, exist_ok=True)
nppes_dataframe.to_parquet(processed_folder_path / "labelled_dataset.parquet", index=False)
logger.info(f"Saved: {processed_folder_path / 'labelled_dataset.parquet'}")

2026-05-17 02:32:09,523 | INFO | Saved: D:\Data Science\Visual Studio Code\healthcare-provider-termination\data\processed\labelled_dataset.parquet


## 5. Summary

```
01_data_ingestion.ipynb
├── Load OIG LEIE        — 82,749 excluded providers
├── Load NPPES sample    — 500,000 providers, 330 columns
├── NPI overlap check    — 1,183 matched, 14.2% (sample rate)
├── Target variable      — excluded = 1 (1,183) / 0 (498,817)
└── Save to parquet      — data/processed/labelled_dataset.parquet
```

**Key finding:** 0.237% exclusion rate — severe class imbalance, must be handled in modelling.

**Why parquet and not CSV?** Parquet is a compressed columnar format — smaller file size, preserves data types.